# Bond Payment Data

<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 1px solid #eaeaea;">
  
  <a href="https://colab.research.google.com/github/PatrickJHess/Volume-Three-Chapter-Three/blob/master/colab/Colab_bond_payment_data.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 28px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Volume-Three-Chapter-Three/master?urlpath=lab/tree/notebooks/bond_payment_data.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">🚀</span> Launch Live in Binder
  </a>

  <a href="https://patrickjhess.github.io/Volume-Three-Chapter-Three/" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">⬅️</span> Return to Main Book
  </a>
</div>

A bond's accrued interest is calculated based on scheduled payment amounts and dates, but its present value requires the actual amounts and dates. Although the payment amounts are fixed, the actual payment dates often deviate from the scheduled ones.

Actual payment dates correspond to settlement dates, while scheduled payments are determined relative to the bond's maturity date. Since settlement days exclude weekends, holidays, and Good Friday, any payments scheduled for these days are advanced to the next available settlement day.
The `bond_payment_data` function in the notebook is used to determine both payment dates and amounts. This function relies on `adjust_bond_pay_dates`, which exemplifies the "Iceberg Principle." It utilizes the pandas_market_calendars library, a powerful tool providing valid trade dates for over fifty exchanges (including NYSE, LSE, and EUREX) and bond calendars for the U.S., U.K., and Japan.

For this notebook, the U.S. calendar, `SIFMAUS`, is employed. The calendar is based on FED bank holidays and incorrectly treats Good Friday as a settlement date. To correct this the Pandas module, `tseries.holidays`and the rule `GoodFriday` are imported from the Pandas `tseries` module. This allows for a modification of the SIFMAUS-created calendar to accurately account for Good Friday, resulting in the correct set of settlement dates for U.S. bonds. The NumPy function `busday_offset` is used to account for non setlement dates, including Good Friday.

The "Iceberg Principle" is further demonstrated in the companion notebook, *Bootstrapping Zero Prices*. That notebook leverages the `bond_payment_data`, `accrued_interest`, `FEDInvest`, and `clean_FEDInvest` functions to calculate zero prices from a sample of coupon bonds.


## Preparing the notebook

:::{important} [ ▼ ] How to use this page: Run, Copy, & Download
:class: dropdown

<ul>
  <li><b>⏻ Run code right here:</b> Click the <b>Power Button</b> icon at the top of the screen to activate <b>Live Code</b>.</li>
  <li><b>📋 Copy code:</b> Hover over any code block and click the <b>Clipboard icon</b> in the top-right corner.</li>
  <li><b>📥 Download this file:</b> Click the <b>Download icon</b> (downward arrow) at the top right of the screen to save this exact notebook to your computer.</li>
</ul>
:::

### Importing libraries, modules, And functions

Modules that are included in the standard Python library are imported. When necessary, other modules or libraries are installed before they are imported. (see [Control Statements](https://patrickjhess.github.io/Introduction-To-Python-For-Financial-Python/Control_Statements.html#the-try-and-except)).

```
import os
import sys
import requests
from datetime import date, datetime
from types import ModuleType

try:
    import pandas as pd
except:
    !pip -q install pandas
    import pandas as pd
    
try:
  import pandas_market_calendars as mcal
except:
  !pip -q install pandas_market_calendars
  import pandas_market_calendars as mcal
  
from pandas.tseries.holiday import GoodFriday
```

In [4]:
import os
import sys
import requests
from datetime import date, datetime
from types import ModuleType
try:
    import pandas as pd
except:
    !pip -q install pandas
    import pandas as pd
try:
  import pandas_market_calendars as mcal
except:
  !pip -q install pandas_market_calendars
  import pandas_market_calendars as mcal
    
from pandas.tseries.holiday import GoodFriday

## Getting `bond_pay_data` from the custom module

:::{important} ☁️ Cloud-Loading: How In-Memory Modules Work
:class: dropdown

**The Logic:**
Usually, Python looks for modules as `.py` files on your hard drive. Here, we are "tricking" Python into treating a string of text from a URL as a live library.

**The Workflow:**
1. **Fetch:** `requests.get(url)` grabs the raw text of your Python script from Dropbox.
2. **Instantiate:** `ModuleType(module_name)` creates an empty "container" in your computer's RAM.
3. **Execute:** `exec(code, module.__dict__)` runs that text inside the container, turning text into live functions.
4. **Register:** By adding it to `sys.modules`, we tell Python: *"If I try to import this later, don't look on the disk—look right here in the memory."*

**Why do this?**
It makes your notebooks **100% portable**. A user can open this in a brand-new environment, and as long as they have an internet connection, all your custom financial functions will "just work."
:::

### Adding a custom module and importing functions


Now that we’ve imported our main modules and libraries, we’ll access a custom module for our more specific functions. In the code below, the custom module `module_basic_concepts_fixed_income` contains functions utilized by this notebook and others in the volume Basic Concepts of Fixed Income.

We access this module from Dropbox using `requests.get()`. This approach allows the notebook to remain "portable"—it fetches the necessary tools directly from the cloud without requiring you to manage local files.

The module is instantiated with `ModuleType` (imported earlier). Once created, the module becomes accessible in the Notebook’s memory, though it is not saved to your hard drive. The `exec()` function then executes the Python code returned by the URL and assigns it to `sys.module`, making the functions ready for use.

Finally, we import the functions `bond_pay_data`.


```
try:
    response = requests.get(url)
    module = ModuleType(module_name)
    exec(response.text, module.__dict__)
    sys.modules[module_name] = module
    # Now we can import from our in-memory module
    from module_basic_concepts_fixed_income import (bond_pay_data)
except requests.exceptions.RequestException as e:
    print(f"❌ Error: Could not fetch module from URL. {e}")
except Exception as e:
    print(f"❌ Error: Failed to execute or import the module. {e}")
```

Our now-complete code is shown in the cell below.

In [2]:
# Define the URL of the Python module to be downloaded from Dropbox.
# The 'dl=1' parameter in the URL forces a direct download of the file content.
url= 'https://www.dropbox.com/scl/fi/4y5hjxlfphh1ngvbgo77q/\
module_-basic_concepts_fixed_income.py?rlkey=6oxi7mgka42veaat79hcv8boz&st=87sztshr&dl=1'
module_name='basic_concepts_fixed_income'
# Send an HTTP GET request to the URL and store the server's response.
try:
    response = requests.get(url)
    module = ModuleType(module_name)
    exec(response.text, module.__dict__)
    sys.modules[module_name] = module
    # Now we can import from our in-memory module
    from basic_concepts_fixed_income import (bond_pay_data)
except requests.exceptions.RequestException as e:
    print(f"❌ Error: Could not fetch module from URL. {e}")
except Exception as e:
    print(f"❌ Error: Failed to execute or import the module. {e}")
    # Now we can import from our in-memory module

## Two new features: `pandas_market_calendars` and the NumPy function `busday_offset`
This notebook explores two new features: the `pandas_market_calendars` library and the `busday_offset` NumPy function. To help you experiment with these concepts, two code snippets are provided.

The first snippet uses the `pandas_market_calendars` library to find non settlement days between Decemember 21<sup>st</sup> 2025 and January 1<sup></sup> 2026.  Christmas and New Year's are detected. Good Friday occurs not occur between the dates.

The second snippet utilizes `busday_offset` to assign a settlement day to both holidays as well as weekends. This snippet assumes that the Datetime indexes `dates`, `fed_holidays_idex`, and `Good_Friday_idx` were created by the first snippet.

````{dropdown} ✂️ Click to see the code snippets

**First Snippet Demonstrates `pandas_market_calendars`**
```python
try:
  import pandas_market_calendars as mcal
except:
  !pip -q install pandas_market_calendars
  import pandas_market_calendars as mcal
from pandas.tseries.holiday import GoodFriday
# create datetime index
start=pd.Timestamp(2025,12,21)
end=pd.Timestamp(2026,1,10)
dates = pd.date_range(start=start, end=end)

# use the library to create the object
fed_cal = mcal.get_calendar('SIFMAUS')
fed_holidays = fed_cal.holidays().holidays

# get Good Fridays
Good_Fridays = GoodFriday.dates(start,end)

# create Pandas Datetime indexes 
fed_holiday_idx = pd.DatetimeIndex(fed_holidays)
Good_Friday_idx = pd.DatetimeIndex(Good_Fridays)

# ceate filter for fed holidays to limit number
fed_holidays_start_end=(fed_holiday_idx>=start) & (fed_holiday_idx<=end) 

# display the results for Panda Datetime indexes
display(fed_holiday_idx[fed_holidays_start_end])
display(Good_Friday_idx)
```
**Second snippet demonstrates NumPy busday_offset**
```python
import numpy as np
# combine fed_and Good Friday Datetime indexes
combined_holidays_idx=fed_holiday_idx.union(Good_Friday_idx)

# Numpy dates must be datetime64
numpy_holidays =combined_holidays_idx.values.astype('datetime64[D]')

# Use NumPy for fully vectorized date math
actual_payment_dates = np.busday_offset(
    dates.values.astype('datetime64[D]'),
    offsets=0,
    roll='forward',
    holidays=numpy_holidays
)

# convert adctual dates to datetime.date
settlement_dates = pd.to_datetime(actual_payment_dates).date

# dates that need adjusting
adjusted_dates=[{actual,settlement}
                         for actual,settlement in zip(dates.date,settlement_dates)
                         if actual!=settlement]
display(adjusted_dates)

````

## **Putting `pandas_market_calendar` and `busday_offset` to work: `adjust_bond_pay_dates`**

The `adjust_bond_pay_dates` function is designed to take one or more dates and return a Pandas DatetimeIndex of corresponding settlement dates. This function relies heavily on the `pandas_market_calendar` library and `busday_offset` of NumPy to perform its core calculations. It is subsequently called by the `bond_pay_data` function that returns the payment amounts and dates for bonds.

````{dropdown} 🔍 Click to see adjust_bond_pay_dates 

```python
def adjust_bond_pay_dates(dates,calendar='SIFMAUS'):
  """
  Adjusts bond payment dates to account for holidays and weekends.
  dates can be a scalar, pandas series, or numpy array (datetime.date,timestamp, or datetime64)
  """

  import pandas as pd
  import numpy as np
  import pandas_market_calendars as mcal
  from pandas.tseries.holiday import GoodFriday

  # Ensure dates is a DatetimeIndex
  if not pd.api.types.is_scalar(dates):
      dates = pd.DatetimeIndex(pd.to_datetime(dates))
  else:
      dates = pd.DatetimeIndex(pd.to_datetime([dates]))

  sifma = mcal.get_calendar(calendar)
  sifma_holidays = sifma.holidays().holidays
  good_fridays = GoodFriday.dates('2000-01-01', '2060-12-31')

  # Convert to DatetimeIndex and use .union() (which automatically deduplicates and sorts)
  sifma_idx = pd.DatetimeIndex(sifma_holidays)
  gf_idx = pd.DatetimeIndex(good_fridays)
  master_bond_holidays = sifma_idx.union(gf_idx)

  # Create a CustomBusinessDay offset using the combined holidays
  numpy_holidays = master_bond_holidays.values.astype('datetime64[D]')

  # Use NumPy for lightning-fast, fully vectorized date math (No warnings!)
  actual_payment_dates = np.busday_offset(
      dates.values.astype('datetime64[D]'),
      offsets=0,
      roll='forward',
      holidays=numpy_holidays
  )

  final_dates = pd.to_datetime(actual_payment_dates).date

  return final_dates 
```
````
````{dropdown} ✂️ Code snippet to see an example of the function.

```python
# import the function from the custom module
from basic_concepts_fixed_income import adjust_bond_pay_dates
# import Numpy to define the dates as an array
try:
    import numpy as np
except:
    !pip -q install numpy
    import numpy as np
try:
  import pandas_market_calendars as mcal
except:
  !pip -q install pandas_market_calendars
  import pandas_market_calendars as mcal
# Good Friday and Easter 2026  
dates=pd.DatetimeIndex([date(2026,3,31),datetime(2026,4,1),datetime(2026,4,2),
             datetime(2026,4,3),datetime(2026,4,5)])
actual_settlement=adjust_bond_pay_dates(dates)
display(actual_settlement)
    
```
````

## **Getting the actual payment dates and amounts**

The `bond_pay_data` function calculates both the payment dates and amounts.  The function requires a maturity date and a coupon.  The settlement date and frequency of payments default to the current day and 2 for semi-annual payments.  The function relies upon the helper functions `scheduled_pay_dates` of Chapter Two of the Volume and `adjust_bond_pay_dates` of this notebook.  The function returns an array of dates and payments. Those dates are used in the next notebook to bootstrap zero prices from coupon bonds.

The `bond_pay_data` function is used to calculate both the payment dates and the corresponding payment amounts for a bond. It requires the bond's maturity date and coupon as inputs. By default, the settlement date is set to the current day, and the payment frequency is set to 2 for semi-annual payments, though these can be specified.

This function relies on two helper functions: `scheduled_pay_dates` (from Chapter Two of the Volume) and `adjust_bond_pay_dates` (found in this notebook).

The output of `bond_pay_data` is an array containing the calculated dates and payments. These dates are then used in the subsequent notebook for bootstrapping zero prices from coupon bonds.

````{dropdown} 🔍 Click to see the <code>bond_pay_data</code> function 

```python
def bond_pay_data(maturity, coupon, settlement=None, freq=2):
    '''
    Function calculates payment Dates And Amounts.
    maturity is a datetime object and coupon is a real number.
    Required arguments are maturity and annual coupon.
    If provided, the value of settlement is a datetime object;
    otherwise defaults to date.today()
    freq defaults to semi-annual but accepts freq equal
    to 1 for annual, 2 for semi-annal, 4 for quarterly, and 12 for monthly.
    The function assumes a par value of 100.
    Returns Numpy arrays of dates and amounts.

    Raises:
        TypeError: If maturity or settlement are not datetime objects.
        ValueError: If inputs are not logically valid (e.g., negative coupon,
                    maturity before settlement).
    '''
    from datetime import datetime, date
    from dateutil.relativedelta import relativedelta
    import pandas as pd
    import numpy as np
    from IPython.display import display, Markdown as md

    # Validate the data - maturity, coupon, settlement, freq
    def validate_date(datetime_object):
        # check for datetime or date
        if not isinstance(datetime_object, (datetime, date)):
            raise TypeError("Input must be a datetime or date object.")
        # convert datetime to date
        if isinstance(datetime_object, datetime):
            datetime_object = datetime_object.date()
        return datetime_object

    # maturity
    maturity = validate_date(maturity)

    # settlement
    if settlement is None:
        settlement = date.today()
    else:
        settlement = validate_date(settlement)

    # coupon
    try:
        coupon = float(coupon)
        if coupon < 0:
            raise ValueError("coupon rate cannot be negative.")
    except (ValueError, TypeError):
        raise ValueError("coupon must be a valid number.")

    # freq
    if int(freq) not in [1, 2, 4, 12]:
        display(md(f"### ⚠️ your assigned freq {freq} it must be (1, 2, 4, or 12)\n ### semi-annual assumed (2)."))
        freq = int(2)

    # check maturity greater than settlement
    if maturity <= settlement:
        raise ValueError("maturity must be greater than the settlement date")

    if coupon == 0:
        # Adjust maturity for non-settlement day and return date and face value
        adjust_maturity = adjust_bond_pay_dates(maturity)
        return np.array([adjust_maturity['Settlement'].dt.date]), np.array([100.0])

    # get scheduled payment dates from helper function scheduled_pay_dates
    scheduled_dates = scheduled_pay_dates(maturity, settlement, freq)

    # Pandas DataFrame Settlement desired column
    both_dates = adjust_bond_pay_dates(scheduled_dates)
    pay_dates=np.array(both_dates['Settlement'].dt.date)
    # calculate payments
    # coupon divided by freq at each date
    pay = np.full(len(pay_dates), coupon / freq)

    # Add principal payment as last cash payment
    pay[-1] += 100

    return pay_dates,pay
```
````



:::::{admonition} ✍️ Application: Calculate the payment dates and amounts for two bonds that mature on August 31, 2035. One bond has a zero coupon and the other an annual coupon of 4.


**The Challenge**:Use the function `bond_pay_data`.


*Recall:* The 'Iceberg Principle' of functions that depend upon other functions that are imported.



---
    
::::{dropdown} 🛟 Need hints or a solution?

:::{dropdown} 💡 Hints

* **Imports:** Refer to the Preparing notebook section of this notebook
* **Create the maturity and settlement dates:** 'date(year,month,day)`



:::

:::{dropdown} ✅ Example Of Solution
## Example of Code


```py
# set the maturity and settlement date
maturity=date(2035,8,31)
settlement=date(2026,4,21)

# iterate throgh the coupons and display results
for coupon in [4,0]:
    display(bond_pay_data(maturity,coupon,settlement=settlement,freq=2))

```
:::
::::
:::::